In [2]:
# nohup python main.py --save_path result --dataset physics --device 0 --data_type=iid --epochs=1000 --model=Diffusion --use_encoder --save_model --eval_interval=40 --lr_decay --n_run=1 --ecp_n_sim=100 --ecp_n_samples=200 > logs/training_physics.log 2>&1 &

In [ ]:
python main.py --save_path result --dataset cos --device 0 --data_type=iid --epochs=1000 --model=Diffusion --use_encoder --save_model --eval_interval=40 --lr_decay --n_run=1 --ecp_n_sim=100 --ecp_n_samples=200

In [3]:
from pathlib import Path
import sys
# Project path setup
project_root = Path.cwd()
if (project_root / 'src').exists():
    pass
else:
    # Fallback: user-specific path
    project_root = Path('/home/chu034/Yaohang_Li/cDiff')


if not project_root.exists() or not (project_root / 'src').exists():
    raise FileNotFoundError(f"找不到项目目录或 src: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"项目路径: {project_root}")
print("Python 路径已更新")



项目路径: /home/chu034/Yaohang_Li/cDiff
Python 路径已更新


In [4]:
import torch
import torch.optim as optim
import numpy as np
from datasets import load_dataset
from models.neural_sampler import NormalizingFlowPosteriorSampler, DiffusionPosteriorSampler
from evaluation.SBC import sample_sbc_calstats, evaluate_sbc
from evaluation.TARP import get_ecp_area_difference
from utils import *
import pandas as pd
import time

In [5]:
use_encoder =1
n_batches = 20      
batch_size = 512
dataset = "cos"
dataset_generator, sample_theta, sample_data = load_dataset(dataset)

if use_encoder:
    dl = dataset_generator(n_batches, batch_size, return_ds=False)
else:
    dl = dataset_generator(n_batches, batch_size, n_sample=1, return_ds=False)
theta, y = next(iter(dl))
y_dim = y.shape[-1]
theta_dim = theta.shape[1]

In [6]:
theta

tensor([[-0.8605,  0.3498],
        [ 0.6627,  0.3655],
        [-0.8107, -0.9313],
        ...,
        [ 0.6944, -0.7485],
        [ 0.7693, -0.6660],
        [-0.7822, -0.1986]])

In [7]:
y

tensor([[[-0.2221],
         [ 0.4573],
         [ 0.3679],
         ...,
         [-0.2632],
         [-0.2394],
         [ 0.1761]],

        [[ 0.4023],
         [ 0.1620],
         [ 0.7240],
         ...,
         [-0.0732],
         [ 0.3573],
         [ 0.4646]],

        [[-0.1933],
         [-0.6211],
         [-0.1107],
         ...,
         [ 0.0087],
         [ 0.3897],
         [ 0.0733]],

        ...,

        [[-0.2451],
         [-0.3720],
         [-0.7057],
         ...,
         [-0.5405],
         [-0.5394],
         [-0.0528]],

        [[-0.6636],
         [-0.8054],
         [-0.0737],
         ...,
         [-0.3570],
         [-0.4304],
         [-0.4712]],

        [[-0.3199],
         [ 0.3225],
         [ 0.0984],
         ...,
         [ 0.2599],
         [-0.2934],
         [ 0.2000]]])

In [8]:
use_encoder =1
n_batches = 20      
batch_size = 512
dataset = "physics"
dataset_generator, sample_theta, sample_data = load_dataset(dataset)

if use_encoder:
    dl= dataset_generator(n_batches, batch_size, return_ds=False)
else:
    dl= dataset_generator(n_batches, batch_size, n_sample=1, return_ds=False)
theta, y = next(iter(dl))
y_dim = y.shape[-1]
theta_dim = theta.shape[1]
dl, ds = dataset_generator(n_batches, batch_size, n_sample=1, return_ds=True)


In [15]:
theta, y = next(iter(dl))


In [9]:
theta

tensor([[ 0.7741, -0.7798,  0.3666,  0.8325,  0.0262,  0.1062],
        [ 0.3499, -0.6035,  0.6631,  0.8954,  0.4440,  0.6775],
        [ 0.5014, -0.2401,  0.4570,  0.1948,  0.7125,  0.8090],
        ...,
        [ 0.6315, -0.6066,  0.4250,  0.8453,  0.5447,  0.1090],
        [ 0.7198, -0.6461,  0.4841,  0.0558,  0.3205,  0.9079],
        [ 0.7822, -0.9427,  0.7078,  0.3217,  0.0808,  0.9291]])

In [10]:
y

tensor([[[0.2089, 0.3148],
         [0.1524, 0.4898],
         [0.2586, 0.3218],
         ...,
         [0.5173, 0.1480],
         [0.1755, 0.2889],
         [0.5413, 0.6643]],

        [[0.3807, 0.5970],
         [0.1358, 0.2456],
         [0.3649, 0.2139],
         ...,
         [0.5226, 0.2541],
         [0.3924, 0.4163],
         [0.4962, 0.3292]],

        [[0.6611, 0.2618],
         [0.1507, 0.4620],
         [0.3118, 0.1545],
         ...,
         [0.1329, 0.6704],
         [0.2446, 0.7668],
         [0.3839, 0.4909]],

        ...,

        [[0.4972, 0.4732],
         [0.8034, 0.5726],
         [0.5919, 0.5280],
         ...,
         [0.4946, 0.3933],
         [0.5659, 0.9273],
         [0.2980, 0.2637]],

        [[0.1708, 0.1107],
         [0.2641, 0.7211],
         [0.4853, 0.2707],
         ...,
         [0.2272, 0.1178],
         [0.8128, 0.3670],
         [0.8570, 0.2717]],

        [[0.1309, 0.3296],
         [0.2924, 0.8171],
         [0.3111, 0.5987],
         ...,
 

In [11]:
from models.neural_sampler import NormalizingFlowPosteriorSampler, DiffusionPosteriorSampler

model = DiffusionPosteriorSampler(y_dim=y_dim, x_dim=theta_dim, n_summaries=256,num_hidden_layer=32,
                                              device="cuda",use_encoder=1, data_type="iid", sigma_data=0.5)


Encoder is for iid data. If not, please check it.


In [12]:
import torch
import torch.optim as optim
from main import trainer
optimizer = optim.Adam(model.parameters(), lr=1e-3)
optimizer_sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

            # Training
model, evaluation_sbc, evaluation_ecp, df_loss, ecp_traj = trainer(dl,ds,model,optimizer,optimizer_sched,20,"cuda",1,1000, 100, 3,
                                         "diffusion", 40,  "/home/chu034/Yaohang_Li/cDiff")

/home/chu034/Yaohang_Li/cDiff/main.py:208: SyntaxWarning: invalid escape sequence '\s'
  parser.add_argument('--use_emperical_sigma', action='store_true', help="whether to set \sigma_data as empirical std of data, otherwise 0.5 as EDM")


ValueError: not enough values to unpack (expected 3, got 2)